In [1]:
from pyspark.sql import SparkSession
import os
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.appName("hw1").getOrCreate()

26/03/03 10:52:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/03 10:52:30 WARN Config: Error reading service account token from: [/var/run/secrets/kubernetes.io/serviceaccount/token]. Ignoring.
26/03/03 10:52:31 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/03 10:52:35 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to event-logs/eventlog_v2_spark-dc2b6480cf324b3997f448d4b68d11f6/events_1_spark-dc2b6480cf324b3997f448d4b68d11f6. This is unsupported


In [3]:
users = spark.createDataFrame(
    [
        ("u1", "Berlin"),
        ("u2", "Berlin"),
        ("u3", "Munich"),
        ("u4", "Hamburg"),
    ],
    ["user_id", "city"]
)

orders = spark.createDataFrame(
    [
        ("o1", "u1", "p1", 2, 10.0),
        ("o2", "u1", "p2", 1, 30.0),
        ("o3", "u2", "p1", 1, 10.0),
        ("o4", "u2", "p3", 5, 7.0),
        ("o5", "u3", "p2", 3, 30.0),
        ("o6", "u3", "p3", 1, 7.0),
        ("o7", "u4", "p1", 10, 10.0),
    ],
    ["order_id", "user_id", "product_id", "qty", "price"]
)

products = spark.createDataFrame(
    [
        ("p1", "Ring VOLA"),
        ("p2", "Ring POROG"),
        ("p3", "Ring TISHINA"),
    ],
    ["product_id", "product_name"]
)

In [4]:
orders_extended = orders.selectExpr(
    "*",
    "qty * price as revenue"
)

In [5]:
joined_dataset = (
    orders_extended.alias("o")
    .join(users.alias("u"), F.col("o.user_id") == F.col("u.user_id"), "inner")
    .join(products.alias("p"), F.col("o.product_id") == F.col("p.product_id"), "inner")
    .select(
        F.col("u.city"),
        F.col("p.product_id"),
        F.col("p.product_name"),
        F.col("o.order_id"),
        F.col("o.qty"),
        F.col("o.revenue")
    )
)

In [6]:
city_product_metrics = (
    joined_dataset
    .groupBy("city", "product_id", "product_name")
    .agg(
        F.count("order_id").alias("orders_cnt"),
        F.sum("qty").alias("qty_sum"),
        F.sum("revenue").alias("revenue_sum")
    )
)

In [7]:
window_by_city = Window.partitionBy("city") \
                        .orderBy(F.desc("revenue_sum"))

top_products_by_city = (
    city_product_metrics
    .withColumn("position", F.row_number().over(window_by_city))
    .filter(F.col("position") <= 2)
    .drop("position")
)

In [9]:
s3_path = "s3a://spark-bucket-test/tmp/sandbox_zeppelin/mart_city_top_products/"
top_products_by_city.write.mode("overwrite").parquet(s3_path)

In [11]:
result_df = spark.read.parquet(s3_path)

result_df.orderBy("city", F.desc("revenue_sum")).show()

+-------+----------+------------+----------+-------+-----------+
|   city|product_id|product_name|orders_cnt|qty_sum|revenue_sum|
+-------+----------+------------+----------+-------+-----------+
| Berlin|        p3|Ring TISHINA|         1|      5|       35.0|
| Berlin|        p1|   Ring VOLA|         2|      3|       30.0|
|Hamburg|        p1|   Ring VOLA|         1|     10|      100.0|
| Munich|        p2|  Ring POROG|         1|      3|       90.0|
| Munich|        p3|Ring TISHINA|         1|      1|        7.0|
+-------+----------+------------+----------+-------+-----------+

